# ARIA v2 Homework 4

This notebook upgrades the Week 3 river-distance workflow with terrain intelligence for **花蓮縣**.

**Deliverables generated by this notebook**

- `terrain_risk_audit.json`
- `terrain_risk_map.png`
- `terrain_risk_top10_scatter.png`

Each major step includes a Captain's Log note so the analysis is easy to follow during grading.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq update
!apt-get -qq install -y fonts-noto-cjk > /dev/null

%pip install -q geopandas rioxarray rasterstats python-dotenv matplotlib pyogrio rasterio shapely xarray


## Captain's Log 1

Lock the Colab paths first. The notebook lives in `Colab Notebooks`, and the raw inputs live in `Colab Notebooks/data`. This version can use either a county-specific DEM or a full Taiwan DEM, then clip it down to the target county before terrain analysis.


In [ ]:
from pathlib import Path
import os
import json

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
import rioxarray
import xarray as xr
from dotenv import load_dotenv
from matplotlib.colors import LightSource
from rasterstats import zonal_stats
from shapely.geometry import mapping

fm._load_fontmanager(try_read_cache=False)
font_candidates = [
    "Noto Sans CJK TC",
    "Noto Sans CJK SC",
    "Noto Sans CJK JP",
    "Microsoft JhengHei",
    "SimHei",
]
available_fonts = {fm.FontProperties(fname=path).get_name() for path in fm.findSystemFonts()}
selected_font = next((font for font in font_candidates if font in available_fonts), None)
if selected_font:
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = [selected_font, "DejaVu Sans"]
    print("Using CJK font:", selected_font)
else:
    print("No CJK font detected; Chinese labels may render with warnings.")
plt.rcParams["axes.unicode_minus"] = False

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs" / "homework4"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENV_PATH = BASE_DIR / ".env"
load_dotenv(ENV_PATH)

TARGET_COUNTY = os.getenv("TARGET_COUNTY", "花蓮縣")
SLOPE_THRESHOLD = float(os.getenv("SLOPE_THRESHOLD", "30"))
ELEVATION_LOW = float(os.getenv("ELEVATION_LOW", "50"))
BUFFER_HIGH = float(os.getenv("BUFFER_HIGH", "500"))
COUNTY_BUFFER = float(os.getenv("COUNTY_BUFFER", "1000"))

env_dem_text = os.getenv("DEM_PATH", "").strip()
dem_candidates = []
if env_dem_text:
    dem_candidates.append(Path(env_dem_text))
dem_candidates.extend(
    [
        DATA_DIR / "DEM_tawiwan_V2025.tif",
        DATA_DIR / "dem_20m_taiwan.tif",
        DATA_DIR / "Hualien_dem_merge.tif",
        DATA_DIR / "dem_20m_hualien.tif",
    ]
)

seen_dem = set()
dem_candidates = [path for path in dem_candidates if not (str(path) in seen_dem or seen_dem.add(str(path)))]
DEM_PATH = next((path for path in dem_candidates if path.exists()), None)

RIVER_SHP_PATH = DATA_DIR / "RIVERPOLY" / "riverpoly" / "riverpoly.shp"
TOWNSHIP_SHP_PATH = DATA_DIR / "鄉(鎮、市、區)界線1140318" / "TOWN_MOI_1140318.shp"
SHELTER_CSV_PATH = DATA_DIR / "避難收容處所點位檔案v9.csv"

assert DEM_PATH is not None, f"DEM file not found. Checked: {dem_candidates}"
assert RIVER_SHP_PATH.exists(), f"River shapefile not found: {RIVER_SHP_PATH}"
assert TOWNSHIP_SHP_PATH.exists(), f"Township shapefile not found: {TOWNSHIP_SHP_PATH}"
assert SHELTER_CSV_PATH.exists(), f"Shelter CSV not found: {SHELTER_CSV_PATH}"

print("Using DEM:", DEM_PATH)
TARGET_COUNTY


## Captain's Log 2

Rebuild only the Week 3 pieces needed for Homework 4: clean shelter coordinates, align CRS to `EPSG:3826`, attach township names, and compute the river-distance category.


In [ ]:
def normalize_name(value):
    text = "" if value is None else str(value)
    return text.replace(" ", "").replace("　", "").replace("台", "臺").strip()


def geometry_union(series):
    union_all = getattr(series, "union_all", None)
    if callable(union_all):
        return union_all()
    return series.unary_union


shelters_raw = pd.read_csv(SHELTER_CSV_PATH, encoding="utf-8")
shelters = pd.DataFrame(
    {
        "shelter_id": shelters_raw.iloc[:, 0].astype(str).str.strip(),
        "county_hint": shelters_raw.iloc[:, 1].fillna("").astype(str).str.strip(),
        "longitude": pd.to_numeric(shelters_raw.iloc[:, 4], errors="coerce"),
        "latitude": pd.to_numeric(shelters_raw.iloc[:, 5], errors="coerce"),
        "name": shelters_raw.iloc[:, 6].fillna("").astype(str).str.strip(),
        "capacity": pd.to_numeric(shelters_raw.iloc[:, 8], errors="coerce").fillna(0).astype(int),
    }
)

valid_mask = shelters["longitude"].notna() & shelters["latitude"].notna() & shelters["longitude"].ne(0) & shelters["latitude"].ne(0)
shelters = shelters.loc[valid_mask].copy()

shelters_gdf = gpd.GeoDataFrame(
    shelters,
    geometry=gpd.points_from_xy(shelters["longitude"], shelters["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:3826")

townships = gpd.read_file(TOWNSHIP_SHP_PATH).to_crs("EPSG:3826")
townships["COUNTYNAME"] = townships["COUNTYNAME"].map(normalize_name)
townships["TOWNNAME"] = townships["TOWNNAME"].map(normalize_name)

land_mask = geometry_union(townships.geometry)
shelters_gdf = shelters_gdf.loc[shelters_gdf.geometry.intersects(land_mask)].copy()

town_join = gpd.sjoin(
    shelters_gdf[["shelter_id", "geometry"]],
    townships[["COUNTYNAME", "TOWNNAME", "TOWNCODE", "geometry"]],
    how="left",
    predicate="intersects",
)
town_join = town_join.sort_values(["shelter_id", "TOWNCODE"]).drop_duplicates("shelter_id")
shelters_gdf = shelters_gdf.merge(
    town_join[["shelter_id", "COUNTYNAME", "TOWNNAME", "TOWNCODE"]],
    on="shelter_id",
    how="left",
)

county_boundary = townships.loc[townships["COUNTYNAME"] == normalize_name(TARGET_COUNTY)].dissolve().reset_index(drop=True)
assert len(county_boundary) == 1, f"County dissolve failed for {TARGET_COUNTY}"
county_buffer = county_boundary.copy()
county_buffer["geometry"] = county_buffer.geometry.buffer(COUNTY_BUFFER)

rivers = gpd.read_file(RIVER_SHP_PATH).to_crs("EPSG:3826")
rivers_in_county = gpd.sjoin(rivers, county_boundary[["geometry"]], how="inner", predicate="intersects")
print(f"rivers_in_county = {len(rivers_in_county)}")
assert len(rivers_in_county) > 0, "River sanity check failed. Please verify that the full river polygons were loaded."

river_union = geometry_union(rivers_in_county.geometry)
target_shelters = shelters_gdf.loc[shelters_gdf["COUNTYNAME"] == normalize_name(TARGET_COUNTY)].copy()
target_shelters["distance_to_river_m"] = target_shelters.geometry.distance(river_union)
target_shelters["river_distance_category"] = np.select(
    [
        target_shelters["distance_to_river_m"] < 500,
        target_shelters["distance_to_river_m"] < 1000,
    ],
    ["<500m", "500-1000m"],
    default=">=1000m",
)

print(target_shelters[["shelter_id", "name", "river_distance_category"]].head())
print(f"target_shelters = {len(target_shelters)}")


## Captain's Log 3

Load the DEM, inspect its metadata, and repair the CRS if the local TIFF does not carry one. If the input is a full Taiwan DEM, clip it by the target county buffer bounds first, then do the precise polygon clip.


In [ ]:
dem = rioxarray.open_rasterio(DEM_PATH, masked=True).squeeze(drop=True)
print("Original shape:", dem.shape)
print("Original CRS:", dem.rio.crs)
print("Original transform:", dem.rio.transform())
print("Original bounds:", dem.rio.bounds())

if dem.rio.crs is None:
    dem = dem.rio.write_crs("EPSG:3826")

if str(dem.rio.crs) != "EPSG:3826":
    dem = dem.rio.reproject("EPSG:3826")

county_minx, county_miny, county_maxx, county_maxy = county_buffer.total_bounds
dem_left, dem_bottom, dem_right, dem_top = dem.rio.bounds()
clip_minx = max(county_minx, dem_left)
clip_miny = max(county_miny, dem_bottom)
clip_maxx = min(county_maxx, dem_right)
clip_maxy = min(county_maxy, dem_top)
assert clip_minx < clip_maxx and clip_miny < clip_maxy, "County buffer does not overlap the DEM bounds."

dem_window = dem.rio.clip_box(minx=clip_minx, miny=clip_miny, maxx=clip_maxx, maxy=clip_maxy)
dem_clipped = dem_window.rio.clip(county_buffer.geometry.apply(mapping), county_buffer.crs, drop=True)
print("Window clip bounds:", (clip_minx, clip_miny, clip_maxx, clip_maxy))
print("Window shape:", dem_window.shape)
print("Clipped shape:", dem_clipped.shape)
print("Clipped CRS:", dem_clipped.rio.crs)
print("Clipped bounds:", dem_clipped.rio.bounds())
print("Elevation range:", float(np.nanmin(dem_clipped.values)), float(np.nanmax(dem_clipped.values)))


## Captain's Log 4

Compute slope from the clipped DEM with `np.gradient(..., 20)` and create a quick terrain diagnostic figure.


In [ ]:
elevation = dem_clipped.values.astype(float)
fill_value = float(np.nanmean(elevation))
elevation_filled = np.where(np.isfinite(elevation), elevation, fill_value)

dy, dx = np.gradient(elevation_filled, 20.0)
slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
slope[~np.isfinite(elevation)] = np.nan

slope_da = xr.DataArray(
    slope,
    coords=dem_clipped.coords,
    dims=dem_clipped.dims,
    attrs={"long_name": "slope_degrees"},
).rio.write_crs(dem_clipped.rio.crs)

print("Slope min / max / mean:", float(np.nanmin(slope)), float(np.nanmax(slope)), float(np.nanmean(slope)))
print("Pixels > 30 degrees:", float(np.nanmean(slope > 30) * 100), "%")

ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(elevation_filled, vert_exag=1, dx=20, dy=20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(elevation, cmap="terrain")
axes[0].set_title("Clipped DEM")
axes[1].imshow(slope, cmap="YlOrRd")
axes[1].set_title("Slope (degrees)")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()


## Captain's Log 5

Build 500m shelter buffers and run zonal statistics for mean elevation, elevation standard deviation, and max slope.


In [ ]:
buffer_gdf = target_shelters.copy()
buffer_gdf["geometry"] = buffer_gdf.geometry.buffer(BUFFER_HIGH)

elevation_nodata = np.where(np.isfinite(elevation), elevation, -9999)
slope_nodata = np.where(np.isfinite(slope), slope, -9999)
affine = dem_clipped.rio.transform()

elev_stats = zonal_stats(
    buffer_gdf.geometry,
    elevation_nodata,
    affine=affine,
    nodata=-9999,
    stats=["mean", "std"],
)
slope_stats = zonal_stats(
    buffer_gdf.geometry,
    slope_nodata,
    affine=affine,
    nodata=-9999,
    stats=["max"],
)

target_shelters["mean_elevation"] = [row.get("mean") for row in elev_stats]
target_shelters["std_elevation"] = [row.get("std") for row in elev_stats]
target_shelters["max_slope"] = [row.get("max") for row in slope_stats]

target_shelters[["shelter_id", "name", "mean_elevation", "std_elevation", "max_slope"]].head()


## Captain's Log 6

Apply the composite risk logic from the assignment and export the audit table.


In [ ]:
def classify_risk(row):
    if row["distance_to_river_m"] < 500 and pd.notna(row["max_slope"]) and row["max_slope"] > SLOPE_THRESHOLD:
        return "very_high"
    if row["distance_to_river_m"] < 500 or (pd.notna(row["max_slope"]) and row["max_slope"] > SLOPE_THRESHOLD):
        return "high"
    if row["distance_to_river_m"] < 1000 and pd.notna(row["mean_elevation"]) and row["mean_elevation"] < ELEVATION_LOW:
        return "medium"
    return "low"


target_shelters["risk_level"] = target_shelters.apply(classify_risk, axis=1)
target_shelters["risk_rank"] = target_shelters["risk_level"].map({"very_high": 0, "high": 1, "medium": 2, "low": 3})

audit_df = (
    target_shelters.sort_values(
        ["risk_rank", "max_slope", "distance_to_river_m", "shelter_id"],
        ascending=[True, False, True, True],
    )[
        [
            "shelter_id",
            "name",
            "COUNTYNAME",
            "TOWNNAME",
            "capacity",
            "distance_to_river_m",
            "river_distance_category",
            "mean_elevation",
            "std_elevation",
            "max_slope",
            "risk_level",
        ]
    ]
    .rename(columns={"COUNTYNAME": "county_name", "TOWNNAME": "town_name"})
    .reset_index(drop=True)
)

audit_path = OUTPUT_DIR / "terrain_risk_audit.json"
with audit_path.open("w", encoding="utf-8") as handle:
    json.dump(audit_df.to_dict(orient="records"), handle, ensure_ascii=False, indent=2)

audit_df.head(10)


## Captain's Log 7

Create the final hillshade map and the Top 10 high-risk scatter plot.


In [ ]:
left, bottom, right, top = dem_clipped.rio.bounds()
extent = (left, right, bottom, top)
risk_colors = {
    "very_high": "#8e0000",
    "high": "#d32f2f",
    "medium": "#f57c00",
    "low": "#388e3c",
}

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(elevation, extent=extent, origin="upper", cmap="terrain", alpha=0.95)
ax.imshow(hillshade, extent=extent, origin="upper", cmap="gray", alpha=0.35)
county_boundary.boundary.plot(ax=ax, color="#102a43", linewidth=1.3)
for level in ["very_high", "high", "medium", "low"]:
    subset = target_shelters[target_shelters["risk_level"] == level]
    if subset.empty:
        continue
    subset.plot(ax=ax, markersize=18, color=risk_colors[level], label=level.replace("_", " ").title(), alpha=0.85)
ax.set_title("ARIA v2 Terrain Risk Map")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.legend(loc="upper right")
map_path = OUTPUT_DIR / "terrain_risk_map.png"
fig.tight_layout()
fig.savefig(map_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

top10 = target_shelters.sort_values(["risk_rank", "max_slope", "distance_to_river_m"], ascending=[True, False, True]).head(10)
fig, ax = plt.subplots(figsize=(10, 7))
for level in ["very_high", "high", "medium", "low"]:
    subset = top10[top10["risk_level"] == level]
    if subset.empty:
        continue
    ax.scatter(subset["mean_elevation"], subset["max_slope"], color=risk_colors[level], s=55, label=level.replace("_", " ").title())
for _, row in top10.iterrows():
    ax.annotate(row["name"][:14], (row["mean_elevation"], row["max_slope"]), fontsize=8)
ax.axhline(SLOPE_THRESHOLD, color="#455a64", linestyle="--", linewidth=1.0, label="Slope Threshold")
ax.set_title("Top 10 Terrain Risk Shelters")
ax.set_xlabel("Mean Elevation (m)")
ax.set_ylabel("Max Slope (degrees)")
ax.legend(loc="upper left")
scatter_path = OUTPUT_DIR / "terrain_risk_top10_scatter.png"
fig.tight_layout()
fig.savefig(scatter_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved:", audit_path)
print("Saved:", map_path)
print("Saved:", scatter_path)


## Captain's Log 8

AI diagnostic log summary:

- `Hualien_dem_merge.tif` may not carry CRS metadata, so the notebook explicitly writes `EPSG:3826` before clipping.
- If zonal statistics return `NaN`, the first checks should be CRS alignment and whether the 500m shelter buffer extends outside the clipped DEM.
- The slope calculation uses `np.gradient(..., 20)` so the pixel spacing matches the 20m DEM resolution.
